In [ ]:
#Inspect the NASA Format

from pathlib import Path

# Find data/raw folder
data_dir = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")

fmt_files = list(data_dir.glob("*.fmt"))
lst_files = list(data_dir.glob("*.lst"))

print("=" * 60)
if fmt_files:
    print(f"📄 Format file found: {fmt_files[0].name}")
    print("\n--- CONTENT OF .fmt FILE ---")
    with open(fmt_files[0], "r") as f:
        print(f.read().strip())
else:
    print(f" No .fmt file found in {data_dir.resolve()}!")

print("=" * 60)
if lst_files:
    print(f"📊 Data file found: {lst_files[0].name}")
    print("\n--- FIRST 5 ROWS OF RAW DATA (.lst) ---")
    with open(lst_files[0], "r") as f:
        for _ in range(5):
            print(f.readline().strip())
else:
    print(f" No .lst file found in {data_dir.resolve()}!")
print("=" * 60)

📄 Format file found: omni2_YRSbPyZjOh.fmt

--- CONTENT OF .fmt FILE ---
FORMAT OF THE SUBSETTED FILE
    
    ITEMS                      FORMAT   
     
 1 YEAR                          I4        
 2 DOY                           I4        
 3 Hour                          I3        
 4 Scalar B, nT                  F6.1      
 5 BX, nT (GSE, GSM)             F6.1      
 6 BY, nT (GSM)                  F6.1      
 7 BZ, nT (GSM)                  F6.1      
 8 SW Plasma Temperature, K      F9.0      
 9 SW Proton Density, N/cm^3     F6.1      
10 SW Plasma Speed, km/s         F6.0      
11 Flow pressure                 F6.2      
12 E elecrtric field             F7.2      
13 Kp index                      I3        
14 Dst-index, nT                 I6        
15 AE-index, nT                  I5
 Data file found: omni2_YRSbPyZjOh.lst

--- FIRST 5 ROWS OF RAW DATA (.lst) ---
2018   1  0   8.2   1.1  -2.3  -5.0   98037.  13.8  381.  3.60   1.91 33    10  330
2018   1  1  10.8   5.9   5.9  

In [ ]:
#Load into Pandas & Create Clean Timestamps

import pandas as pd
import numpy as np

# 1. Define clean, human-readable column names matching the .fmt file
column_names = [
    "Year", "DOY", "Hour",
    "Scalar_B", "BX_GSM", "BY_GSM", "BZ_GSM",
    "Temperature", "Proton_Density", "Flow_Speed",
    "Flow_Pressure", "Electric_Field",
    "Kp_index", "Dst_index", "AE_index"
]

# 2. Get the .lst file path
lst_file = lst_files[0]

# 3. Read whitespace-delimited ASCII data
df_raw = pd.read_csv(lst_file, sep=r"\s+", header=None, names=column_names)

# 4. Construct an exact Datetime timestamp using Year, Day of Year (DOY), and Hour
# Note: %j is the standard format code for Day of Year (1-365)
df_raw["Timestamp"] = pd.to_datetime(
    df_raw["Year"].astype(str) + "-" + df_raw["DOY"].astype(str) + " " + df_raw["Hour"].astype(str),
    format="%Y-%j %H"
)

# 5. Set Timestamp as index and drop separate year/doy/hour columns
df_raw.set_index("Timestamp", inplace=True)
df = df_raw.drop(columns=["Year", "DOY", "Hour"])

print(f" Successfully loaded {len(df):,} hourly observations!")
print(f" Time range: from {df.index.min()} to {df.index.max()}")
print("\n--- FIRST 5 ROWS ---")
df.head()

 Successfully loaded 52,584 hourly observations!
 Time range: from 2018-01-01 00:00:00 to 2023-12-31 23:00:00

--- FIRST 5 ROWS ---


,Scalar_B,BX_GSM,BY_GSM,BZ_GSM,Temperature,Proton_Density,Flow_Speed,Flow_Pressure,Electric_Field,Kp_index,Dst_index,AE_index
Timestamp,,,,,,,,,,,,
2018-01-01 00:00:00,8.2,1.1,-2.3,-5.0,98037.0,13.8,381.0,3.60,1.91,33,10,330
2018-01-01 01:00:00,10.8,5.9,5.9,-6.1,92897.0,16.8,406.0,4.94,2.48,33,6,370
2018-01-01 02:00:00,10.0,1.1,8.9,-0.3,83702.0,17.7,392.0,4.85,0.12,33,-1,349
2018-01-01 03:00:00,10.9,2.1,9.2,-2.5,70701.0,22.4,396.0,6.22,0.99,37,0,395
2018-01-01 04:00:00,11.0,-1.3,-1.9,2.4,154791.0,19.0,416.0,5.97,-1.00,37,-7,225


In [ ]:
#Expose the Trap & Replace Fill Values with NaN

# 1. Define NASA's documented fill values for each parameter
fill_values = {
    "Scalar_B": 999.9,
    "BX_GSM": 999.9,
    "BY_GSM": 999.9,
    "BZ_GSM": 999.9,
    "Temperature": 9999999.0,
    "Proton_Density": 999.9,
    "Flow_Speed": 9999.0,
    "Flow_Pressure": 99.99,
    "Electric_Field": 999.99,
    "Kp_index": 99,
    "Dst_index": 99999,
    "AE_index": 9999
}

# 2. Check how many fake "fill values" exist before cleaning
print(" Number of NASA fill values detected in each column:")
for col, fill_val in fill_values.items():
    count = (df[col] == fill_val).sum()
    if count > 0:
        print(f"  • {col:15s}: {count:,} missing records encoded as {fill_val}")

# 3. Create a clean copy and replace fill values with real NaN (Not-a-Number)
df_clean = df.copy()
for col, fill_val in fill_values.items():
    df_clean[col] = df_clean[col].replace(fill_val, np.nan)

# 4. Show missing value percentage summary
print("\n Missing Data Percentage after replacing with NaN:")
missing_pct = (df_clean.isnull().mean() * 100).round(2)
print(missing_pct[missing_pct > 0].to_string())

🔍 Number of NASA fill values detected in each column:
  • Scalar_B       : 265 missing records encoded as 999.9
  • BX_GSM         : 265 missing records encoded as 999.9
  • BY_GSM         : 265 missing records encoded as 999.9
  • BZ_GSM         : 265 missing records encoded as 999.9
  • Temperature    : 457 missing records encoded as 9999999.0
  • Proton_Density : 525 missing records encoded as 999.9
  • Flow_Speed     : 441 missing records encoded as 9999.0
  • Flow_Pressure  : 529 missing records encoded as 99.99
  • Electric_Field : 487 missing records encoded as 999.99

📊 Missing Data Percentage after replacing with NaN:
Scalar_B          0.50
BX_GSM            0.50
BY_GSM            0.50
BZ_GSM            0.50
Temperature       0.87
Proton_Density    1.00
Flow_Speed        0.84
Flow_Pressure     1.01
Electric_Field    0.93


In [ ]:
#Interpolate Gaps & Verify Physical Statistics

# 1. Apply time-based linear interpolation for small sensor gaps
df_clean_imputed = df_clean.interpolate(method="time").bfill().ffill()

# 2. Verify that zero NaNs remain
remaining_nans = df_clean_imputed.isnull().sum().sum()
print(f" Remaining missing values in the entire dataset: {remaining_nans}")

# 3. Check realistic physical statistics
print("\n Physical Summary Statistics (Min, Mean, Max):")
stats_cols = ["Scalar_B", "BZ_GSM", "Flow_Speed", "Proton_Density", "Dst_index"]
print(df_clean_imputed[stats_cols].describe().round(2))

# 4. Save the cleaned dataset to data/processed for ML and Looker Studio!
processed_path = Path("../data/processed/omni2_cleaned.csv") if Path("../data/processed").exists() else Path("data/processed/omni2_cleaned.csv")
df_clean_imputed.to_csv(processed_path)
print(f"\n Cleaned dataset saved to: {processed_path.resolve()}")
print(f"   Shape: {df_clean_imputed.shape[0]:,} rows × {df_clean_imputed.shape[1]} columns")

 Remaining missing values in the entire dataset: 0

📈 Physical Summary Statistics (Min, Mean, Max):
       Scalar_B    BZ_GSM  Flow_Speed  Proton_Density  Dst_index
count  52584.00  52584.00    52584.00        52584.00   52584.00
mean       5.25     -0.13      410.32            6.31      -8.51
std        2.59      2.66       85.65            4.73      14.40
min        0.60    -32.40      259.00            0.10    -213.00
25%        3.60     -1.50      345.00            3.29     -15.00
50%        4.70     -0.20      391.00            5.00      -6.00
75%        6.20      1.20      461.00            7.90       0.00
max       37.40     31.30      846.00           84.40      42.00

 Cleaned dataset saved to: C:\Users\Akanksha\OneDrive\Desktop\ML_Project\data\processed\omni2_cleaned.csv
   Shape: 52,584 rows × 12 columns
